# 08 — PPO with Symbolic (RAM) Observations

PPO agent using the RAM-based symbolic grid (13x16) instead of pixels.

**Setup:**
- Observation: flattened 13x16 grid with 4 frame stack = 832-dim vector
- Policy: MlpPolicy with [512, 512] hidden layers
- Much faster than pixel-based (no CNN, no image processing)
- Same PPO hyperparameters as pixel version

In [1]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

/home/contente/Documents/ENSTA/autonomous/CSC-52081-ContinousMountainCar


In [2]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using CUDA: True
GPU: NVIDIA GeForce GTX 1650


In [3]:
# Create symbolic environment
env = make_symbolic_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4,
    n_stack=4,
    flatten=True,
)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')

Observation space: (832,)
Action space: 7


In [4]:
# Phase 1: Train from scratch
config = PPOConfig(
    policy='MlpPolicy',
    learning_rate=2.5e-4,
    n_steps=512,
    batch_size=256,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    total_timesteps=1_000_000,
)

model = PPO(
    policy=config.policy,
    env=env,
    learning_rate=config.learning_rate,
    n_steps=config.n_steps,
    batch_size=config.batch_size,
    n_epochs=config.n_epochs,
    gamma=config.gamma,
    gae_lambda=config.gae_lambda,
    clip_range=config.clip_range,
    ent_coef=config.ent_coef,
    vf_coef=config.vf_coef,
    max_grad_norm=config.max_grad_norm,
    policy_kwargs=dict(net_arch=[512, 512]),
    tensorboard_log='../logs/symbolic_ppo',
    verbose=1,
)

print(f'Phase 1: {config.total_timesteps:,} timesteps')
print(f'Device: {model.device}')

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Phase 1: 1,000,000 timesteps
Device: cuda


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [5]:
%load_ext tensorboard
%tensorboard --logdir ../logs/symbolic_ppo

In [ ]:
# Phase 1: Train for 1M steps
callback = CheckpointAndLogCallback(
    save_path='../models/symbolic_ppo',
    save_freq=50_000,
)

model.learn(
    total_timesteps=config.total_timesteps,
    callback=callback,
    log_interval=1,
)
model.save('../models/symbolic_ppo/phase1_model')
print('Phase 1 complete!')

Logging to ../logs/symbolic_ppo/PPO_4


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 63       |
|    ep_rew_mean     | 222      |
| time/              |          |
|    fps             | 118      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 512      |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 63         |
|    ep_rew_mean          | 222        |
|    flag_rate_100        | 0          |
|    mean_length_100      | 63         |
|    mean_reward_100      | 222        |
| time/                   |            |
|    fps                  | 115        |
|    iterations           | 2          |
|    time_elapsed         | 8          |
|    total_timesteps      | 1024       |
| train/                  |            |
|    approx_kl            | 0.00784141 |
|    clip_fraction        | 0.0239     |
|    clip_range           | 0.2        |
|   

In [ ]:
# Phase 2: Load best model, lower lr, train 1M more
from stable_baselines3 import PPO

model = PPO.load('../models/symbolic_ppo/phase1_model', env=env)
model.learning_rate = 1e-5

callback2 = CheckpointAndLogCallback(
    save_path='../models/symbolic_ppo',
    save_freq=50_000,
)

model.learn(
    total_timesteps=1_000_000,
    callback=callback2,
    log_interval=1,
)
model.save('../models/symbolic_ppo/final_model')
print('Phase 2 complete!')

In [ ]:
# Plot training results
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

rewards = callback.episode_rewards + (callback2.episode_rewards if 'callback2' in dir() else [])
lengths = callback.episode_lengths + (callback2.episode_lengths if 'callback2' in dir() else [])
flags = callback.episode_flags + (callback2.episode_flags if 'callback2' in dir() else [])

if len(rewards) > 0:
    window = min(100, len(rewards))

    ax = axes[0]
    ax.plot(rewards, alpha=0.3, color='blue')
    if len(rewards) > window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed, color='blue', linewidth=2)
    ax.set_title('Episode Rewards')
    ax.set_xlabel('Episode')

    ax = axes[1]
    ax.plot(lengths, alpha=0.3, color='orange')
    if len(lengths) > window:
        smoothed = np.convolve(lengths, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(lengths)), smoothed, color='orange', linewidth=2)
    ax.set_title('Episode Lengths')
    ax.set_xlabel('Episode')

    ax = axes[2]
    if len(flags) > 0:
        cumulative_flags = np.cumsum(flags) / (np.arange(len(flags)) + 1)
        ax.plot(cumulative_flags, color='green', linewidth=2)
    ax.set_title('Cumulative Flag Rate')
    ax.set_xlabel('Episode')
    ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('../models/symbolic_ppo/training_curves.png', dpi=150)
    plt.show()
else:
    print('No episode data collected yet.')

In [ ]:
# Evaluate
from src.utils.evaluation import evaluate_agent

eval_env = make_symbolic_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4, n_stack=4, flatten=True,
)
results = evaluate_agent(model, eval_env, n_episodes=30)
print(f"Mean reward: {results['mean_reward']:.1f} +/- {results['std_reward']:.1f}")
print(f"Flag rate: {results['flag_rate']:.2%}")